# Zebrafish: GitHub + scripts (API + SRA Toolkit)

We keep the **same repo** locally and on `sequoia:/home/zebrafish` so everyone runs the same scripts.
We use **two download approaches**: API for metadata/coordination, and SRA Toolkit for FASTQ generation.


## Why we keep both approaches

API = fast exploration + reproducible run lists; SRA Toolkit = standard implementation for producing FASTQs (fastq-dump / fasterq-dump).


## 0) Confirm local vs server repo are in sync

This prints the git commit hash locally and on the server (`/home/zebrafish`). If SSH fails, run `ssh pzg8794@sequoia.rit.edu` in a terminal once to set up keys/hostkey.



In [ ]:
!ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 -n pzg8794@sequoia.rit.edu "hostname; whoami; pwd"


Checks SSH connectivity to the server (should print hostname, username, and a working directory).


In [ ]:
%%bash
set -euo pipefail

# Compare local vs server git HEAD
GIT_ROOT="$(git rev-parse --show-toplevel)"
cd "$GIT_ROOT"

echo "LOCAL HEAD:  $(git rev-parse HEAD)"
git status -sb || true

echo
SSH="ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"
$SSH "REMOTE_REPO=$REMOTE_REPO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

echo "SERVER HEAD: $(git rev-parse HEAD)"
git status -sb || true
EOF


Compares local vs server git commit (`HEAD`) so you know both environments are using the same code.


## 1) Pull the latest repo on the server (fixes missing scripts)

Run this once when things look out of date or you see a “missing file” error.


In [ ]:
%%bash
set -euo pipefail

# Update server repo
SSH="ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

$SSH "REMOTE_REPO=$REMOTE_REPO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

# If git refuses due to 'dubious ownership', run once:
#   git config --global --add safe.directory /home/zebrafish

git fetch origin
# Use ff-only to avoid accidental merges on the shared server

git pull --ff-only

echo "SERVER HEAD: $(git rev-parse HEAD)"
EOF


Updates the shared server clone in `/home/zebrafish` with a fast-forward-only `git pull`.


## 2) What scripts exist (shared by all members)

These are the scripts everyone should use (local or on the server).


In [ ]:
%%bash
set -euo pipefail

# List scripts locally (this notebook) + on server.

echo "LOCAL scripts (cwd=$(pwd))"
ls -la scripts | sed -n '1,200p'

echo
echo "SERVER scripts (/home/zebrafish/zebrafish/scripts)"
SSH="ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

if ! $SSH "REMOTE_REPO=$REMOTE_REPO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"
ls -la zebrafish/scripts | sed -n '1,200p'
EOF
then
  echo
  echo "SSH failed (or server path missing). Run this once in a terminal to fix auth/hostkey:"
  echo "  ssh pzg8794@sequoia.rit.edu"
fi


Lists the scripts locally and on the server so the team knows what is available and in-sync.


## API approach: metadata + coordination

We use the SRA RunInfo API to get run metadata and generate stable SRR lists for the team.


### A1) Script: `get_zebrafish_data_sra.py`

Fetches RunInfo (`runinfo.csv`) and writes SRR lists (all + filtered) under `zebrafish/metadata/<ACC>/`.


In [ ]:
%%bash
set -euo pipefail

# Show script help (server)
SSH="ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

$SSH "REMOTE_REPO=$REMOTE_REPO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"
python3 zebrafish/scripts/get_zebrafish_data_sra.py --help | sed -n '1,120p'
EOF


Shows help for the API metadata script that generates `runinfo.csv` and run lists used by the team.


### A2) Script: `download_runfiles_ncbi_download_path.py`

Downloads the run file for each SRR using the `download_path` column in `runinfo.csv` (no SRA Toolkit needed).


In [ ]:
%%bash
set -euo pipefail

# Show script help (server)
SSH="ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

$SSH "REMOTE_REPO=$REMOTE_REPO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"
python3 zebrafish/scripts/download_runfiles_ncbi_download_path.py --help | sed -n '1,160p'
EOF


Shows help for the script that downloads SRA “runfiles” via NCBI RunInfo `download_path`.

Example (server):

```bash
python3 zebrafish/scripts/download_runfiles_ncbi_download_path.py \
  --acc PRJNA1277581 \
  --runinfo-csv zebrafish/metadata/PRJNA1277581/runinfo.csv \
  --runs-file  zebrafish/metadata/PRJNA1277581/runs.team.txt \
  --base-dir   zebrafish/data/runfiles
```


## SRA Toolkit approach: FASTQs (implementation)

We install SRA Toolkit once, then use it to convert SRRs into paired FASTQ files for analysis.


### S1) Install toolkit: `ensure_sratoolkit.py` (or Step 4a in the other notebook)

This keeps the toolkit in `zebrafish/tools/sratoolkit/` (gitignored) so server + local usage matches.


In [ ]:
%%bash
set -euo pipefail

# Install / ensure SRA Toolkit on the server
SSH="ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

$SSH "REMOTE_REPO=$REMOTE_REPO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"
python3 zebrafish/scripts/ensure_sratoolkit.py --repo-root "$PWD" --print-bin
EOF


Ensures SRA Toolkit exists under `zebrafish/tools/sratoolkit/` on the server and prints its `bin/` path.


### S2) Script: `download_fastq_sratoolkit_from_runs.sh`

Downloads FASTQs from a **runs file** (use `runs.team.txt` or your member split) using `prefetch` + `fasterq-dump --split-files --threads N`.


In [ ]:
%%bash
set -euo pipefail

# Show script help (server)
SSH="ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

$SSH "REMOTE_REPO=$REMOTE_REPO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

bash zebrafish/scripts/download_fastq_sratoolkit_from_runs.sh --help | sed -n '1,200p'
EOF


Shows help for the SRA Toolkit FASTQ downloader script (takes a runs file + output directory).

Examples (server):

```bash
# Team-wide download
bash zebrafish/scripts/download_fastq_sratoolkit_from_runs.sh \
  --runs-file zebrafish/metadata/PRJNA1277581/runs.team.txt \
  --out-dir   zebrafish/data/PRJNA1277581/team \
  --threads   4

# Per-member download (after Step 3 split)
bash zebrafish/scripts/download_fastq_sratoolkit_from_runs.sh \
  --runs-file zebrafish/metadata/PRJNA1277581/splits/runs.member.piter.txt \
  --out-dir   zebrafish/data/PRJNA1277581 \
  --threads   4
```


## How we use the professor’s SRA Toolkit commands (and how we wrap them)

Our FASTQ download is just the professor’s workflow automated for a **list of SRR accessions**. The wrapper script is `zebrafish/scripts/download_fastq_sratoolkit_from_runs.sh` and it runs the same two SRA Toolkit commands for each SRR:

### 1) `prefetch <SRR>` (download the run into the SRA cache)
- What it does: downloads the run file for an accession (an `.sra`-style runfile) into the SRA Toolkit cache (typically under `~/.ncbi/public/sra/`, depending on toolkit config).
- Why we use it: it’s **resumable** and keeps the “download” step separate from conversion, so if conversion fails you don’t re-download everything.

### 2) `fasterq-dump ... <SRR>` (convert runfile → FASTQ)
In our wrapper we call:

```bash
fasterq-dump --split-files --threads <N> --outdir <RUN_DIR> <SRR>
```

- `--split-files`: for paired-end runs, writes two files: `<SRR>_1.fastq` and `<SRR>_2.fastq`.
- `--threads <N>`: uses multiple threads to speed up conversion.
- `--outdir <RUN_DIR>`: writes outputs into a per-run folder so files from different SRRs never collide.

### 3) `gzip -f` (compress FASTQs)
`fasterq-dump` writes **uncompressed** `.fastq` by default, so we immediately compress:

```bash
gzip -f <SRR>_1.fastq <SRR>_2.fastq
```

That produces the final files we keep:
- `<SRR>_1.fastq.gz`
- `<SRR>_2.fastq.gz`

### How the wrapper script ties it together
For each SRR in `--runs-file`, the script:
1. Creates `--out-dir/<SRR>/`
2. **Skips** the SRR if `<SRR>_1.fastq.gz` and `<SRR>_2.fastq.gz` already exist (unless `--force` is used)
3. Runs `prefetch <SRR>`
4. Runs `fasterq-dump --split-files --threads ... --outdir ... <SRR>`
5. Runs `gzip -f` on the two FASTQs

This gives us a reproducible, team-friendly pattern: everyone runs the same script with a different runs list (or split list), but the underlying commands are exactly the professor’s.

### Quick note: `fastq-dump` vs `fasterq-dump`
- For **full downloads**, we prefer `prefetch + fasterq-dump` because it’s the modern/fast path.
- In the *other* notebook where we extract a **spot-range subset**, we use `fastq-dump -N/-X` because `fastq-dump` supports spot subsetting while the toolkit build we have may not support `-N/-X` on `fasterq-dump`.


## 3) Choose runs for team download

Pick runs by (a) a number of runs, (b) a runs file, or (c) an inline list; we split this once for the team.


In [ ]:
%%bash
set -euo pipefail

# Build a single team run list (server) that everyone will use.
# Then split that list among piter/nikhi/samuel (no per-member filtering).

SSH="ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

ACC="PRJNA1277581"

# Choose ONE input method (server-side):
N_RUNS=0                  # 0 = use all runs from runs.all.txt
RUNS_FILE=""              # e.g., zebrafish/metadata/$ACC/runs.some_list.txt
RUNS_INLINE=""            # e.g., "SRR123 SRR456 SRR789" (space-separated)

$SSH "REMOTE_REPO=$REMOTE_REPO ACC=$ACC N_RUNS=$N_RUNS RUNS_FILE=$RUNS_FILE RUNS_INLINE=$RUNS_INLINE bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

ACC="$ACC"
N_RUNS="$N_RUNS"
RUNS_FILE="$RUNS_FILE"
RUNS_INLINE="$RUNS_INLINE"

ALL="zebrafish/metadata/$ACC/runs.all.txt"
TEAM="zebrafish/metadata/$ACC/runs.team.txt"
SPLITS_DIR="zebrafish/metadata/$ACC/splits"
mkdir -p "$(dirname "$TEAM")" "$SPLITS_DIR"

if [ -n "$RUNS_INLINE" ]; then
  echo "$RUNS_INLINE" | tr ' ' $'
' | sed '/^$/d' > "$TEAM"
  echo "TEAM runs from inline list -> $TEAM"
elif [ -n "$RUNS_FILE" ]; then
  [ -f "$RUNS_FILE" ] || { echo "ERROR: RUNS_FILE not found: $RUNS_FILE"; exit 2; }
  cp "$RUNS_FILE" "$TEAM"
  echo "TEAM runs from file ($RUNS_FILE) -> $TEAM"
else
  [ -f "$ALL" ] || { echo "ERROR: missing $ALL (run the RunInfo step first)"; exit 2; }
  if [ "$N_RUNS" != "0" ]; then
    head -n "$N_RUNS" "$ALL" > "$TEAM"
    echo "TEAM runs = first $N_RUNS from $ALL -> $TEAM"
  else
    cp "$ALL" "$TEAM"
    echo "TEAM runs = all runs from $ALL -> $TEAM"
  fi
fi

echo
echo "TEAM run count + preview:"
wc -l "$TEAM" || true
head -n 5 "$TEAM" || true

python3 zebrafish/scripts/split_runs_among_members.py   --runs-file "$TEAM"   --members piter nikhi samuel   --out-dir "$SPLITS_DIR"   --prefix runs.member

echo
echo "Split files:"
ls -la "$SPLITS_DIR"
EOF


Creates `runs.team.txt` (team-wide) then splits it into per-member run lists under `.../splits/`.


## Member download: piter

Set `DO_RUN=1` to actually start downloading FASTQs for piter’s assigned SRRs.


In [ ]:
%%bash
set -euo pipefail

# piter: download FASTQs on the server (SRA Toolkit)
SSH="ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

ACC="PRJNA1277581"
THREADS=4

# Smoke test: download ONLY 1 run first (recommended)
N_RUNS=1

$SSH "REMOTE_REPO=$REMOTE_REPO ACC=$ACC THREADS=$THREADS N_RUNS=$N_RUNS MEMBER=piter bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

export PATH="$PWD/zebrafish/tools/sratoolkit/bin:$PATH"

ACC="$ACC"
THREADS="$THREADS"
N_RUNS="$N_RUNS"
MEMBER="$MEMBER"

RUNS_FILE_BASE="zebrafish/metadata/$ACC/splits/runs.member.${MEMBER}.txt"
OUT_DIR_BASE="zebrafish/data/$ACC"

[ -f "$RUNS_FILE_BASE" ] || { echo "ERROR: missing $RUNS_FILE_BASE (run Step 3)"; exit 2; }
mkdir -p "$OUT_DIR_BASE" "zebrafish/metadata/$ACC/splits"

# --- Example (RUNS = N_RUNS = 1) ---
RUNS_FILE_USE="zebrafish/metadata/$ACC/splits/runs.member.${MEMBER}.first${N_RUNS}.txt"
OUT_DIR="$OUT_DIR_BASE"

head -n "$N_RUNS" "$RUNS_FILE_BASE" > "$RUNS_FILE_USE"

echo "runs_file_base: $RUNS_FILE_BASE"
echo "runs_file_use:  $RUNS_FILE_USE"
echo "out_dir:        $OUT_DIR"
echo "threads:        $THREADS"

echo
echo "Starting download (N_RUNS=$N_RUNS)..."
bash zebrafish/scripts/download_fastq_sratoolkit_from_runs.sh   --runs-file "$RUNS_FILE_USE"   --out-dir   "$OUT_DIR"   --threads   "$THREADS"

# Cleanup: this notebook cell is a demo (keeps the shared data folder clean)
SRR_DEMO="$(head -n 1 "$RUNS_FILE_USE" | tr -d $'\r' | xargs)"
echo
echo "Demo finished. Cleaning up SRR=$SRR_DEMO"
rm -rf "$OUT_DIR/$SRR_DEMO"
rm -f "$RUNS_FILE_USE"

# --- Other common variants (keep commented until you need them) ---

# 1) Download your FULL assigned list:
# bash zebrafish/scripts/download_fastq_sratoolkit_from_runs.sh #   --runs-file "$RUNS_FILE_BASE" #   --out-dir   "$OUT_DIR_BASE" #   --threads   "$THREADS"

# 2) Download from an arbitrary runs file you provide:
# RUNS_FILE_ANY="zebrafish/metadata/$ACC/runs.some_list.txt"
# bash zebrafish/scripts/download_fastq_sratoolkit_from_runs.sh #   --runs-file "$RUNS_FILE_ANY" #   --out-dir   "$OUT_DIR_BASE" #   --threads   "$THREADS"

# 3) Download from an inline list (space-separated SRRs):
# RUNS_INLINE="SRR34002427 SRR34002428"
# RUNS_FILE_INLINE="zebrafish/metadata/$ACC/splits/runs.inline.${MEMBER}.txt"
# echo "$RUNS_INLINE" | tr ' ' $'\n' > "$RUNS_FILE_INLINE"
# bash zebrafish/scripts/download_fastq_sratoolkit_from_runs.sh \

#   --runs-file "$RUNS_FILE_INLINE" \
#   --out-dir   "$OUT_DIR_BASE" \
#   --threads   "$THREADS"
EOF


Runs a **1-run smoke test** for Piter using his split run list; other variants are included but commented out.


## Member download: nikhi

Set `DO_RUN=1` to actually start downloading FASTQs for nikhi’s assigned SRRs.


In [ ]:
%%bash
set -euo pipefail

# nikhi: download FASTQs on the server (SRA Toolkit)
SSH="ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

ACC="PRJNA1277581"
THREADS=4

# Smoke test: download ONLY 1 run first (recommended)
N_RUNS=1

$SSH "REMOTE_REPO=$REMOTE_REPO ACC=$ACC THREADS=$THREADS N_RUNS=$N_RUNS MEMBER=nikhi bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

export PATH="$PWD/zebrafish/tools/sratoolkit/bin:$PATH"

ACC="$ACC"
THREADS="$THREADS"
N_RUNS="$N_RUNS"
MEMBER="$MEMBER"

RUNS_FILE_BASE="zebrafish/metadata/$ACC/splits/runs.member.${MEMBER}.txt"
OUT_DIR_BASE="zebrafish/data/$ACC"

[ -f "$RUNS_FILE_BASE" ] || { echo "ERROR: missing $RUNS_FILE_BASE (run Step 3)"; exit 2; }
mkdir -p "$OUT_DIR_BASE" "zebrafish/metadata/$ACC/splits"

# --- Example (RUNS = N_RUNS = 1) ---
RUNS_FILE_USE="zebrafish/metadata/$ACC/splits/runs.member.${MEMBER}.first${N_RUNS}.txt"
OUT_DIR="$OUT_DIR_BASE"

head -n "$N_RUNS" "$RUNS_FILE_BASE" > "$RUNS_FILE_USE"

echo "runs_file_base: $RUNS_FILE_BASE"
echo "runs_file_use:  $RUNS_FILE_USE"
echo "out_dir:        $OUT_DIR"
echo "threads:        $THREADS"

echo
echo "Starting download (N_RUNS=$N_RUNS)..."
bash zebrafish/scripts/download_fastq_sratoolkit_from_runs.sh   --runs-file "$RUNS_FILE_USE"   --out-dir   "$OUT_DIR"   --threads   "$THREADS"

# Cleanup: this notebook cell is a demo (keeps the shared data folder clean)
SRR_DEMO="$(head -n 1 "$RUNS_FILE_USE" | tr -d $'\r' | xargs)"
echo
echo "Demo finished. Cleaning up SRR=$SRR_DEMO"
rm -rf "$OUT_DIR/$SRR_DEMO"
rm -f "$RUNS_FILE_USE"

# --- Other common variants (keep commented until you need them) ---

# 1) Download your FULL assigned list:
# bash zebrafish/scripts/download_fastq_sratoolkit_from_runs.sh #   --runs-file "$RUNS_FILE_BASE" #   --out-dir   "$OUT_DIR_BASE" #   --threads   "$THREADS"

# 2) Download from an arbitrary runs file you provide:
# RUNS_FILE_ANY="zebrafish/metadata/$ACC/runs.some_list.txt"
# bash zebrafish/scripts/download_fastq_sratoolkit_from_runs.sh #   --runs-file "$RUNS_FILE_ANY" #   --out-dir   "$OUT_DIR_BASE" #   --threads   "$THREADS"

# 3) Download from an inline list (space-separated SRRs):
# RUNS_INLINE="SRR34002427 SRR34002428"
# RUNS_FILE_INLINE="zebrafish/metadata/$ACC/splits/runs.inline.${MEMBER}.txt"
# echo "$RUNS_INLINE" | tr ' ' $'\n' > "$RUNS_FILE_INLINE"
# bash zebrafish/scripts/download_fastq_sratoolkit_from_runs.sh \

#   --runs-file "$RUNS_FILE_INLINE" \
#   --out-dir   "$OUT_DIR_BASE" \
#   --threads   "$THREADS"
EOF


Runs a **1-run smoke test** for Nikhi using his split run list; other variants are included but commented out.


## Member download: samuel

Set `DO_RUN=1` to actually start downloading FASTQs for samuel’s assigned SRRs.


In [ ]:
%%bash
set -euo pipefail

# samuel: download FASTQs on the server (SRA Toolkit)
SSH="ssh -o StrictHostKeyChecking=accept-new -o BatchMode=yes -o ConnectTimeout=10 -o ServerAliveInterval=5 -o ServerAliveCountMax=1 pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

ACC="PRJNA1277581"
THREADS=4

# Smoke test: download ONLY 1 run first (recommended)
N_RUNS=1

$SSH "REMOTE_REPO=$REMOTE_REPO ACC=$ACC THREADS=$THREADS N_RUNS=$N_RUNS MEMBER=samuel bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

export PATH="$PWD/zebrafish/tools/sratoolkit/bin:$PATH"

ACC="$ACC"
THREADS="$THREADS"
N_RUNS="$N_RUNS"
MEMBER="$MEMBER"

RUNS_FILE_BASE="zebrafish/metadata/$ACC/splits/runs.member.${MEMBER}.txt"
OUT_DIR_BASE="zebrafish/data/$ACC"

[ -f "$RUNS_FILE_BASE" ] || { echo "ERROR: missing $RUNS_FILE_BASE (run Step 3)"; exit 2; }
mkdir -p "$OUT_DIR_BASE" "zebrafish/metadata/$ACC/splits"

# --- Example (RUNS = N_RUNS = 1) ---
RUNS_FILE_USE="zebrafish/metadata/$ACC/splits/runs.member.${MEMBER}.first${N_RUNS}.txt"
OUT_DIR="$OUT_DIR_BASE"

head -n "$N_RUNS" "$RUNS_FILE_BASE" > "$RUNS_FILE_USE"

echo "runs_file_base: $RUNS_FILE_BASE"
echo "runs_file_use:  $RUNS_FILE_USE"
echo "out_dir:        $OUT_DIR"
echo "threads:        $THREADS"

echo
echo "Starting download (N_RUNS=$N_RUNS)..."
bash zebrafish/scripts/download_fastq_sratoolkit_from_runs.sh   --runs-file "$RUNS_FILE_USE"   --out-dir   "$OUT_DIR"   --threads   "$THREADS"

# Cleanup: this notebook cell is a demo (keeps the shared data folder clean)
SRR_DEMO="$(head -n 1 "$RUNS_FILE_USE" | tr -d $'\r' | xargs)"
echo
echo "Demo finished. Cleaning up SRR=$SRR_DEMO"
rm -rf "$OUT_DIR/$SRR_DEMO"
rm -f "$RUNS_FILE_USE"

# --- Other common variants (keep commented until you need them) ---

# 1) Download your FULL assigned list:
# bash zebrafish/scripts/download_fastq_sratoolkit_from_runs.sh #   --runs-file "$RUNS_FILE_BASE" #   --out-dir   "$OUT_DIR_BASE" #   --threads   "$THREADS"

# 2) Download from an arbitrary runs file you provide:
# RUNS_FILE_ANY="zebrafish/metadata/$ACC/runs.some_list.txt"
# bash zebrafish/scripts/download_fastq_sratoolkit_from_runs.sh #   --runs-file "$RUNS_FILE_ANY" #   --out-dir   "$OUT_DIR_BASE" #   --threads   "$THREADS"

# 3) Download from an inline list (space-separated SRRs):
# RUNS_INLINE="SRR34002427 SRR34002428"
# RUNS_FILE_INLINE="zebrafish/metadata/$ACC/splits/runs.inline.${MEMBER}.txt"
# echo "$RUNS_INLINE" | tr ' ' $'\n' > "$RUNS_FILE_INLINE"
# bash zebrafish/scripts/download_fastq_sratoolkit_from_runs.sh \

#   --runs-file "$RUNS_FILE_INLINE" \
#   --out-dir   "$OUT_DIR_BASE" \
#   --threads   "$THREADS"
EOF


Runs a **1-run smoke test** for Samuel using his split run list; other variants are included but commented out.
